### Replace the citation numbers in a saved Perplexity dialogue with matching literature note or zotero item links

In [1]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

In [2]:
tmp_dir = rfw.refwrangle_test_dir / 'tmp'

perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
obsidian_citekeys_file = rfw.refwrangle_test_dir / "dat" / 'obsnotecitekeys.csv'

output_file = tmp_dir / "tmp_new_cites_perplexity_example.md"

##### get the URLs of all parent items in the zotero db, and find out which have obsidian literature notes


In [3]:
zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)
parentItems = zot.everything(zot.top())

In [4]:
# Make a lookup dict: zotero DB item URL to bibtex citekey
citekeys = {}
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)

    if 'url' in pdat:
        purl = rfw.normalize_url(pdat['url'])
        if len(purl)>0:
            citekeysForURL[purl].append(citekeyThis)

repeatedURLs = {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}

if (nURLrepeats := len(repeatedURLs)) > 0:
    print(f"There were {nURLrepeats} URLs with > 1 parent (citekey)")
    for url in repeatedURLs.keys():
        print(f"{repeatedURLs[url]}\n\t{url}")
    raise Exception(f'Not built for repeated URLS: {nURLrepeats=}.')

url_to_citekey={}
for (key, value_list) in citekeysForURL.items():
    url_to_citekey[key] = value_list[0]

In [5]:
# Collect info about each zotero DB item that has a URL

lit_note_file_stems = {fNm.stem for fNm in rfw.lit_notes_obsidian_dir.glob('*.md')}

citekeys = {}
zot_db_items = []
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)
    zot_db_items.append(dict(citekey=citekeyThis, zotkey=parent['key'], hasLitNote=citekeyThis in lit_note_file_stems))

    if 'url' in pdat:
        purl = rfw.normalize_url(pdat['url'])
        if len(purl)>0:
            citekeysForURL[purl].append(citekeyThis)

repeatedURLs = {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}

if (nURLrepeats := len(repeatedURLs)) > 0:
    print(f"There were {nURLrepeats} URLs with > 1 parent (citekey)")
    for url in repeatedURLs.keys():
        print(f"{repeatedURLs[url]}\n\t{url}")
    raise Exception(f'Not built for repeated URLS: {nURLrepeats=}.')

url_to_citekey={}
citekey_to_url = {}
for (url, citekey_list) in citekeysForURL.items():
    url_to_citekey[url] = citekey_list[0]
    citekey_to_url[citekey_list[0]] = url


zot_db_items = pd.DataFrame(zot_db_items).set_index('citekey')
zot_db_items['url'] = pd.Series(citekey_to_url)
zot_db_items = zot_db_items.reset_index()

hasNoURL = zot_db_items.url.isna()
if (nURLmiss := sum(hasNoURL)) > 0:
    print(f"Dropping {nURLmiss=} of {len(zot_db_items)} zotero entries which have no URL")
    zot_db_items_no_url = zot_db_items[hasNoURL]
    zot_db_items = zot_db_items[~hasNoURL]
    display(zot_db_items_no_url)

Dropping nURLmiss=165 of 1680 zotero entries which have no URL


,citekey,zotkey,hasLitNote,url
16,Seals99irradFrcctDiag,WYP9J7EU,False,NaN
73,LaPaglia13TestIncrsSuggestibility,LDTF7M3L,False,NaN
134,Gaur20attribModellingRvw,HLHKVCLX,False,NaN
138,Holloway23emotionCuePolitJudge,4AUIK74F,False,NaN
146,Tomeo21predElectAgeSocMedia,PX6DNXY2,False,NaN
...,...,...,...,...
1672,Mayhorn16disaggLdRealWrldPerf,RCMMDNV6,False,NaN
1676,Heinemann06frcstSolRadCMV,NYJFP3I4,False,NaN
1677,Lorenz07frcstEnsGridPV,2C5N2DZC,False,NaN
1678,Perez18newSolFrcstSiteSpec,NJCJWR59,False,NaN


In [6]:
zot_db_items

,citekey,zotkey,hasLitNote,url
0,Krysiak-Adamczyk25brandSentimAnlyss,839Z6XEL,False,https://survicate.com/blog/brand-sentiment-ana...
1,Wolanin24brandSentimentCare,7A9XQEFG,False,https://brand24.com/blog/brand-sentiment
2,Mohamed2monitorToolsPR,MW8W5LIA,False,https://www.aimtechnologies.co/pr-monitoring-t...
3,Mohamed23sentimAnlyssTools,XZ927L88,False,https://www.aimtechnologies.co/sentiment-analy...
4,Mathew21socialMediaToolsTop23,K3EGPXPI,False,https://www.meltwater.com/en/blog/top-social-m...
...,...,...,...,...
1669,Oracle19evDetDisaggAMI,JI47UFCR,False,https://www.oracle.com/a/ocom/docs/industries/...
1671,Bidgely19amInsightsRprt,EKLZCYP5,False,https://www.idcutilitiessummit.com/index/resou...
1673,Hare18disaggHmLdDmdResp,WA8IQAXP,False,https://dspace.mit.edu/handle/1721.1/117983
1674,Rehman21LoadDisaggThesis,8MAAZN8P,False,https://openrepository.aut.ac.nz/handle/10292/...


In [7]:
# Functions for replacing references in perplexity's dialog copy with links to existing obsidian notes or zotero items 

def zotero_item_link(zotero_item_key, link_text):
    """Makes a link to a zotero item, given its key"""
    return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

def normalize_url(url):
    """Convert a URL to a standard form, so tha it can be string-compared to the same URL
    written by a different program, but which is also normalized by this function."""
    parsed = urlparse(url.lower())
    return urlunparse(parsed._replace(path=parsed.path.rstrip('/')))

def replace_perplexity_dialogue_links(perplexity_doc, zot_db_items, output_file):
    """Replace numeric citations in a perplexity dialog document with links to matching 
    obsidian literature notes or to zotero items.  A 'match' is determined when the URL 
    in the perplexity doc matches a zotero item's URL.  Link first to the obsidian literature note
    when one exists, then try to link to a zotero item.  If neither is available don't change the link.

    Arguments 
    perplexity_doc: a full pathlib path to a file of markdown coming from perplexity's copy function
    zot_db_items: a dataframe with a row of info for every zotero DB item.  The columns are: 
        citekey: the obsidian note citekey (the stem of its filename)
        zotkey: zoter item key
        hasLitNote: true if an obsidian literature note already exists
        url: the URL associated with this zotero DB item
    output_file: a full pathlib path to where the output document should go"""

    # organize the zotero DB info
    if not isinstance(zot_db_items, pd.DataFrame):
        raise Exception('Expected a dataframe.  Reading url_to_citekey from file does not yet handle new dataframe column')
        df = pd.read_csv(zot_db_items) # assume it has url and citekey columns
        zot_db_items = {normalize_url(url): citekey for url, citekey in zip(df.url, df.citekey)}

    zot_db_items['url'] = zot_db_items['url'].apply(normalize_url)
    zot_url_to_item_info = defaultdict(lambda: None, {url:info.iloc[0] for url, info in zot_db_items.groupby('url')})

    # modify the perplexity dialog doc
    with open(perplexity_doc, 'r') as mdfile:
        content = mdfile.read()

    # Split the content into body and citations
    parts = content.split("\nCitations:\n")
    if len(parts) != 2:
        raise Exception("Couldn't find Citations section")
    
    body, citations = parts

    # From citations at doc bottom, get a url for each citation number
    citation_urls = re.findall(r'\[(\d+)\]\s+(https?://\S+)', citations)
    doc_number_to_url = defaultdict(lambda: None, {num:normalize_url(url) for num, url in citation_urls})

    def make_best_reference_link(doc_url, doc_cite_num):
        # Replace body citations w/ wikilinks to an obsidian note or if no note, an md link to a zotero item
        if doc_url and (itemInfo := zot_url_to_item_info[doc_url]) is not None:
            if itemInfo.hasLitNote:
                return f'[[{itemInfo.citekey}]]' # wikilink to obsidian lit note

            # md link to item in zotero DB
            link_text = f'{itemInfo.citekey}\u2794{itemInfo.zotkey}'  #"bob \u2794 jim"
            return f'{zotero_item_link(itemInfo.zotkey, link_text)}'
            
        return f'[{doc_cite_num}]' # not in zotero DB so leave unchanged

    def replace_body_reference(match):
        # Replace citations in the body text
        doc_cite_num = match.group(1)
        doc_url = doc_number_to_url[doc_cite_num]

        return ' ' + make_best_reference_link(doc_url, doc_cite_num)

    body = re.sub(r'\[(\d+)\]', replace_body_reference, body)

    def replace_citations_reference(match):
        # Replace citations in the Citations section
        doc_cite_num = match.group(1)
        url = match.group(2)
        doc_url = normalize_url(url)

        if zot_url_to_item_info[doc_url] is None:
            return f'[{doc_cite_num}] {doc_url}' # not in zotero DB
        else:
            return f'[{doc_cite_num}] =={make_best_reference_link(doc_url, doc_cite_num)}== {url}'

    citations = re.sub(r'\[(\d+)\]\s+(https?://\S+)', replace_citations_reference, citations)

    with open(output_file, 'w') as outfile:
        outfile.write(body + "\nCitations:\n" + citations)


In [8]:
replace_perplexity_dialogue_links(perplexity_dialog_file, zot_db_items, output_file)
#rfw.ORIG_replace_perplexity_citations_from_perplexity(perplexity_dialog_file, url_to_citekey, output_file)
print('Done.')

Done.
